# 03 — RAG Evaluation
**PaperSloth — Retrieval Quality + Faithfulness + Ablation Study**

```
Evaluation plan:
  1. Setup & connections
  2. Build evaluation dataset (30 manually-labelled queries)
  3. Precision@5 — hybrid vs dense-only (ablation)
  4. RAGAS — Faithfulness + Answer Relevancy (no ground truth needed)
  5. Results summary + export
```

**What we're evaluating:**
- PaperSloth is a *question retrieval* RAG system — documents are exam questions, not answers
- Retrieval quality (Precision@5): did we surface the right questions for a topic query?
- Faithfulness: does the LLM response only contain info grounded in retrieved context?
- Answer Relevancy: does the response actually address what the student asked?
- Ablation: hybrid (alpha=0.7) vs dense-only (alpha=1.0) to validate architecture decision

In [ ]:
%pip install ragas

---
## Section 1 — Setup & Connections

In [1]:
import os, json, pickle, time
from pathlib import Path
from dotenv import load_dotenv
load_dotenv()

keys = ["GEMINI_API_KEY", "PINECONE_API_KEY", "DATABASE_URL"]
for k in keys:
    print(f"{k}: {'✅' if os.getenv(k) else '❌ MISSING'}")

GEMINI_API_KEY: ✅
PINECONE_API_KEY: ✅
DATABASE_URL: ✅


In [2]:
import ollama
import psycopg2
import google.generativeai as genai
from pinecone import Pinecone
from sentence_transformers import CrossEncoder

# ── Clients ───────────────────────────────────────────────────────────────────
genai.configure(api_key=os.getenv("GEMINI_API_KEY"))
pc    = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
index = pc.Index("papersloth")

# ── Models ────────────────────────────────────────────────────────────────────
EMBED_MODEL   = "nomic-embed-text-v2-moe:latest"
GEMINI_MODEL  = "gemma-4-31b-it"
FLASH_MODEL   = "gemini-3.1-flash-lite"   
RERANKER_PATH = "cross-encoder/ms-marco-MiniLM-L-6-v2"
BM25_PATH     = "data/bm25_model.pkl"

# ── Load BM25 ─────────────────────────────────────────────────────────────────
assert Path(BM25_PATH).exists(), f"BM25 not found at {BM25_PATH}"
with open(BM25_PATH, "rb") as f:
    bm25 = pickle.load(f)

# ── Load reranker ─────────────────────────────────────────────────────────────
reranker = CrossEncoder(RERANKER_PATH)

# ── Gemini generation model ───────────────────────────────────────────────────
RAG_SYSTEM = (
    "You are a UTP past year exam assistant. "
    "You will be given exam questions and a student query. "
    "Output ONLY the matching question text with its marks and source. "
    "Do not explain, reason, or analyse. "
    "Begin your response immediately with the answer."
)
gen_model = genai.GenerativeModel(GEMINI_MODEL, system_instruction=RAG_SYSTEM)

# ── Pinecone stats ────────────────────────────────────────────────────────────
stats = index.describe_index_stats()
print(f"✅ All clients ready")
print(f"   Pinecone: {stats['total_vector_count']} vectors")

/var/folders/my/2fkdjjvj7h9dn3nmk1ttlccc0000gn/T/ipykernel_9943/713641070.py:3: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

✅ All clients ready
   Pinecone: 158 vectors


---
## Section 2 — Core Retrieval Helpers

Extracted from `services/retrieval/` so this notebook is self-contained.

In [3]:
def embed_query(text: str) -> list[float]:
    resp = ollama.embed(model=EMBED_MODEL, input=f"search_query: {text}")
    return resp["embeddings"][0]


def hybrid_scale(dense, sparse, alpha=0.7):
    return (
        [v * alpha for v in dense],
        {"indices": sparse["indices"], "values": [v * (1 - alpha) for v in sparse["values"]]},
    )


def fetch_parents(parent_ids: list[str]) -> list[dict]:
    if not parent_ids:
        return []
    conn = psycopg2.connect(os.getenv("DATABASE_URL"))
    cur  = conn.cursor()
    cur.execute("""
        SELECT parent_id, question_number, full_text, total_marks,
               course_code, semester, year
        FROM   parent_chunks
        WHERE  parent_id = ANY(%s)
    """, (parent_ids,))
    rows = cur.fetchall()
    cur.close(); conn.close()
    return [
        {
            "parent_id":       r[0],
            "question_number": r[1],
            "full_text":       r[2],
            "total_marks":     r[3],
            "course_code":     r[4],
            "semester":        r[5],
            "year":            r[6],
        }
        for r in rows
    ]


def retrieve(
    query: str,
    course_code: str = None,
    top_k: int = 20,
    rerank_top_n: int = 5,
    alpha: float = 0.7,      # 0.7 = hybrid, 1.0 = dense-only
) -> list[dict]:
    """
    Run the full retrieval pipeline and return top parent chunks.
    alpha=0.7 → hybrid (production), alpha=1.0 → dense-only (baseline).
    """
    dense  = embed_query(query)
    sparse = bm25.encode_queries([query])[0]
    d, s   = hybrid_scale(dense, sparse, alpha)

    pinecone_filter = {"course_code": {"$eq": course_code}} if course_code else None

    results = index.query(
        vector           = d,
        sparse_vector    = s,
        top_k            = top_k,
        include_metadata = True,
        filter           = pinecone_filter,
    )

    if not results.matches:
        return []

    # Rerank
    passages = [m.metadata.get("text_preview", "") for m in results.matches]
    scores   = reranker.predict([(query, p) for p in passages])
    ranked   = sorted(zip(results.matches, scores), key=lambda x: x[1], reverse=True)
    top      = [m for m, _ in ranked[:rerank_top_n]]

    # Parent expansion
    parent_ids = list({m.metadata["parent_id"] for m in top})
    return fetch_parents(parent_ids)


def generate_answer(query: str, parents: list[dict]) -> str:
    """LLM generation over retrieved context — used for RAGAS metrics."""
    if not parents:
        return "No relevant questions found."
    parts = []
    for doc in parents:
        header = (
            f"[{doc['course_code']} | {doc['semester']} {doc['year']} | "
            f"Q{doc['question_number']} | {doc['total_marks']} marks]"
        )
        parts.append(f"{header}\n{doc['full_text']}")
    context = "\n\n---\n\n".join(parts)
    prompt  = f"EXAM QUESTIONS:\n{context}\n\nSTUDENT: {query}"
    resp = gen_model.generate_content(prompt, generation_config={"temperature": 0.1})
    return resp.text


print("✅ Retrieval helpers ready")

✅ Retrieval helpers ready


---
## Section 3 — Evaluation Dataset

**30 manually-labelled topic queries.**

Each entry has:
- `query`: what a student would type
- `relevant_question_numbers`: list of question numbers that ARE relevant (you label these)
- `course_code`: subject to filter on (or None for cross-subject)

**How to label:** run the query mentally against what you know is in your papers.
A question is relevant if a student searching this topic would genuinely want that question.

⚠️ **Be strict.** Only mark as relevant if the question directly covers the topic.
Tangentially related questions should NOT be marked relevant.

In [ ]:
# ── EVALUATION DATASET ────────────────────────────────────────────────────────
# Fill in relevant_question_numbers based on what you know is in your papers.
# question_number matches the `question_number` column in parent_chunks.
#
# Format: {"query", "course_code", "relevant_question_numbers", "query_type"}
# query_type: topic_search | fetch | trend — for breakdown analysis

EVAL_DATASET = [
    # ── RBB3013 Sept 2025 (verified from your ingestion output) ──────────────
    {
        "query": "thermocouple temperature voltage measurement",
        "course_code": "RBB3013",
        "relevant_question_numbers": ["2"],   # Q2 contains Q2c.i and Q2c.ii
        "query_type": "topic_search",
    },
    {
        "query": "LRV URV pressure vessel level measurement",
        "course_code": "RBB3013",
        "relevant_question_numbers": ["2"],
        "query_type": "topic_search",
    },
    {
        "query": "PI controller zero offset feedback loop",
        "course_code": "RBB3013",
        "relevant_question_numbers": ["4"],
        "query_type": "topic_search",
    },
    {
        "query": "standard deviation average voltage error analysis",
        "course_code": "RBB3013",
        "relevant_question_numbers": ["1"],
        "query_type": "topic_search",
    },
    {
        "query": "PMMC voltmeter series resistance calculation",
        "course_code": "RBB3013",
        "relevant_question_numbers": ["1"],
        "query_type": "topic_search",
    },
    {
        "query": "flow control loop instrument diagram",
        "course_code": "RBB3013",
        "relevant_question_numbers": ["3"],
        "query_type": "topic_search",
    },
    {
        "query": "absolute gauge differential pressure terminology",
        "course_code": "RBB3013",
        "relevant_question_numbers": ["3"],
        "query_type": "topic_search",
    },
    {
        "query": "wet leg level measurement requirement",
        "course_code": "RBB3013",
        "relevant_question_numbers": ["2"],
        "query_type": "topic_search",
    },
    {
        "query": "5G wireless transmission industrial system",
        "course_code": "RBB3013",
        "relevant_question_numbers": ["3"],
        "query_type": "topic_search",
    },
    {
        "query": "d'Arsonval movement full scale current multirange",
        "course_code": "RBB3013",
        "relevant_question_numbers": ["1"],
        "query_type": "topic_search",
    },

    # ── Add your remaining papers below ──────────────────────────────────────
    # Replace the placeholders with real queries from your ingested papers.
    # The more papers you have, the more diverse your eval set.
    #
    # Template:
    # {
    #     "query": "<topic from your paper>",
    #     "course_code": "<code>",
    #     "relevant_question_numbers": ["<qnum>"],
    #     "query_type": "topic_search",
    # },

    # ── Cross-subject queries (no course_code filter) ─────────────────────────
    {
        "query": "thermocouple type J voltage millivolts",
        "course_code": None,
        "relevant_question_numbers": ["2"],   # from RBB3013
        "query_type": "topic_search",
    },
    {
        "query": "PI controller transfer function closed loop",
        "course_code": None,
        "relevant_question_numbers": ["4"],
        "query_type": "topic_search",
    },

    # ── Hard / negative queries (should return nothing or low precision) ──────
    # These test that you don't return garbage for unrelated topics.
    # relevant_question_numbers = [] means nothing should be retrieved.
    {
        "query": "neural network deep learning backpropagation",
        "course_code": "RBB3013",
        "relevant_question_numbers": [],
        "query_type": "negative",
    },
    {
        "query": "database SQL joins foreign key",
        "course_code": "RBB3013",
        "relevant_question_numbers": [],
        "query_type": "negative",
    },
]

print(f"Evaluation dataset: {len(EVAL_DATASET)} queries")
print(f"  topic_search : {sum(1 for q in EVAL_DATASET if q['query_type'] == 'topic_search')}")
print(f"  negative     : {sum(1 for q in EVAL_DATASET if q['query_type'] == 'negative')}")
print()
print("⚠️  If you have more papers ingested, add more queries above before running Section 4.")
print("   Aim for 25-30 total for a credible evaluation.")

---
## Section 4 — Precision@5 Ablation

Run each query twice:
- **Hybrid** (alpha=0.7): your production system
- **Dense-only** (alpha=1.0): baseline, no BM25

**Precision@K** = (relevant retrieved in top K) / K

A retrieved parent chunk is *relevant* if its `question_number` is in the query's `relevant_question_numbers`.

This is the core ablation study validating your hybrid architecture decision.

In [ ]:
K = 5   # Precision@K

def precision_at_k(retrieved: list[dict], relevant_qnums: list[str], k: int = 5) -> float:
    """
    What fraction of the top-K retrieved questions are relevant?
    relevant_qnums: list of question_number strings e.g. ['2', '4']
    retrieved: list of parent dicts (already top-k from retrieval)
    """
    if not relevant_qnums:
        # Negative query — precision is 1.0 if nothing retrieved, 0.0 if something is
        return 1.0 if len(retrieved) == 0 else 0.0

    top_k   = retrieved[:k]
    hits    = sum(1 for r in top_k if r["question_number"] in relevant_qnums)
    return hits / min(k, max(len(top_k), 1))


print("✅ precision_at_k defined")
print(f"   Evaluating Precision@{K}")

In [ ]:
# ── Run ablation ──────────────────────────────────────────────────────────────
# This will take a few minutes — each query hits Pinecone + reranker twice.

results = []

for i, item in enumerate(EVAL_DATASET):
    query   = item["query"]
    course  = item["course_code"]
    rel     = item["relevant_question_numbers"]
    qtype   = item["query_type"]

    print(f"[{i+1:02d}/{len(EVAL_DATASET)}] {query[:55]}...")

    # Hybrid (production)
    t0           = time.time()
    hybrid_ret   = retrieve(query, course_code=course, rerank_top_n=K, alpha=0.7)
    hybrid_time  = time.time() - t0
    hybrid_p     = precision_at_k(hybrid_ret, rel, k=K)

    # Dense-only (baseline)
    t0           = time.time()
    dense_ret    = retrieve(query, course_code=course, rerank_top_n=K, alpha=1.0)
    dense_time   = time.time() - t0
    dense_p      = precision_at_k(dense_ret, rel, k=K)

    results.append({
        "query":             query,
        "course_code":       course,
        "query_type":        qtype,
        "relevant_qnums":    rel,
        "hybrid_precision":  hybrid_p,
        "dense_precision":   dense_p,
        "hybrid_retrieved":  [r["question_number"] for r in hybrid_ret],
        "dense_retrieved":   [r["question_number"] for r in dense_ret],
        "hybrid_time_s":     round(hybrid_time, 2),
        "dense_time_s":      round(dense_time, 2),
    })

    delta = hybrid_p - dense_p
    symbol = "✅" if delta >= 0 else "⚠️"
    print(f"       hybrid={hybrid_p:.2f}  dense={dense_p:.2f}  Δ={delta:+.2f} {symbol}")

print("\n✅ Ablation complete")

In [ ]:
# ── Aggregate results ─────────────────────────────────────────────────────────
import statistics

hybrid_scores = [r["hybrid_precision"] for r in results]
dense_scores  = [r["dense_precision"]  for r in results]

# Overall
mean_hybrid = statistics.mean(hybrid_scores)
mean_dense  = statistics.mean(dense_scores)
improvement = (mean_hybrid - mean_dense) / max(mean_dense, 1e-9) * 100

# Per query type
topic_results    = [r for r in results if r["query_type"] == "topic_search"]
negative_results = [r for r in results if r["query_type"] == "negative"]

print("=" * 55)
print("PRECISION@5 RESULTS")
print("=" * 55)
print(f"  Hybrid (alpha=0.7)  : {mean_hybrid:.3f}")
print(f"  Dense-only (α=1.0)  : {mean_dense:.3f}")
print(f"  Improvement         : {improvement:+.1f}%")
print()
print(f"  Topic queries ({len(topic_results)})")
if topic_results:
    th = statistics.mean(r["hybrid_precision"] for r in topic_results)
    td = statistics.mean(r["dense_precision"]  for r in topic_results)
    print(f"    Hybrid : {th:.3f}")
    print(f"    Dense  : {td:.3f}")
print()
print(f"  Negative queries ({len(negative_results)})")
if negative_results:
    nh = statistics.mean(r["hybrid_precision"] for r in negative_results)
    nd = statistics.mean(r["dense_precision"]  for r in negative_results)
    print(f"    Hybrid : {nh:.3f}  (1.0 = correctly returned nothing)")
    print(f"    Dense  : {nd:.3f}")
print("=" * 55)

In [ ]:
# ── Per-query breakdown table ─────────────────────────────────────────────────
print(f"{'#':<4} {'Query':<45} {'Type':<14} {'Hybrid':>7} {'Dense':>7} {'Δ':>6}")
print("-" * 85)
for i, r in enumerate(results):
    delta = r["hybrid_precision"] - r["dense_precision"]
    flag  = "↑" if delta > 0 else ("↓" if delta < 0 else "=")
    print(
        f"{i+1:<4} {r['query'][:44]:<45} {r['query_type']:<14} "
        f"{r['hybrid_precision']:>7.2f} {r['dense_precision']:>7.2f} "
        f"{delta:>+5.2f}{flag}"
    )

---
## Section 5 — RAGAS: Faithfulness + Answer Relevancy

These metrics don't need ground truth answers — they evaluate the LLM generation layer.

- **Faithfulness**: is the generated answer grounded in the retrieved context?
  - High score → LLM only states what is in the retrieved questions
  - Low score → LLM is hallucinating question text or details

- **Answer Relevancy**: does the answer actually respond to what was asked?
  - High score → student asked about thermocouples, answer is about thermocouples
  - Low score → answer drifts off topic

We run this on the topic_search queries only (not negatives).

In [ ]:
# Install ragas if needed
# !pip install ragas --quiet

In [ ]:
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from datasets import Dataset

# RAGAS needs LangChain-wrapped LLM + embeddings
ragas_llm = LangchainLLMWrapper(
    ChatGoogleGenerativeAI(
        model="gemini-2.0-flash-lite",
        google_api_key=os.getenv("GEMINI_API_KEY"),
        temperature=0,
    )
)
ragas_emb = LangchainEmbeddingsWrapper(
    GoogleGenerativeAIEmbeddings(
        model="models/text-embedding-004",
        google_api_key=os.getenv("GEMINI_API_KEY"),
    )
)

print("✅ RAGAS LLM + embeddings ready")

In [ ]:
# ── Build RAGAS dataset ───────────────────────────────────────────────────────
# Only run on topic_search queries (not negatives — no context to evaluate)
#
# RAGAS EvaluationDataset format:
#   user_input    : the student query
#   retrieved_contexts : list of context strings (the full_text of parents)
#   response      : what the LLM generated

ragas_rows = []
topic_queries = [item for item in EVAL_DATASET if item["query_type"] == "topic_search"]

print(f"Building RAGAS dataset for {len(topic_queries)} topic queries...")
print("(This calls Gemini once per query — may take ~2 min)")
print()

for i, item in enumerate(topic_queries):
    query  = item["query"]
    course = item["course_code"]

    print(f"[{i+1:02d}/{len(topic_queries)}] {query[:55]}...")

    # Use hybrid retrieval (production system)
    parents = retrieve(query, course_code=course, rerank_top_n=5, alpha=0.7)

    if not parents:
        print("       ⚠️  No results — skipping")
        continue

    # Generate LLM response
    response = generate_answer(query, parents)

    # Contexts = full_text of each retrieved parent
    contexts = [p["full_text"] for p in parents]

    ragas_rows.append({
        "user_input":          query,
        "retrieved_contexts":  contexts,
        "response":            response,
    })

print(f"\n✅ RAGAS dataset built: {len(ragas_rows)} samples")

In [ ]:
# ── Run RAGAS evaluation ──────────────────────────────────────────────────────
ragas_dataset = Dataset.from_list(ragas_rows)

ragas_results = evaluate(
    dataset = ragas_dataset,
    metrics = [faithfulness, answer_relevancy],
    llm     = ragas_llm,
    embeddings = ragas_emb,
)

print("✅ RAGAS evaluation complete")
print()
print(ragas_results)

In [ ]:
# ── Per-sample RAGAS scores ───────────────────────────────────────────────────
ragas_df = ragas_results.to_pandas()
ragas_df["query_short"] = ragas_df["user_input"].str[:50]

display_cols = ["query_short", "faithfulness", "answer_relevancy"]
print(ragas_df[display_cols].to_string(index=False))
print()
print(f"Mean Faithfulness    : {ragas_df['faithfulness'].mean():.3f}")
print(f"Mean Answer Relevancy: {ragas_df['answer_relevancy'].mean():.3f}")

---
## Section 6 — Final Summary

All metrics in one place. Copy these numbers to your resume / report.

In [ ]:
# ── Collect all numbers ───────────────────────────────────────────────────────
mean_faithfulness     = ragas_df["faithfulness"].mean()
mean_answer_relevancy = ragas_df["answer_relevancy"].mean()

print("=" * 60)
print("PAPERSLOTH — RAG EVALUATION SUMMARY")
print("=" * 60)
print()
print("  RETRIEVAL QUALITY (Precision@5)")
print(f"    Hybrid dense-sparse (α=0.7) : {mean_hybrid:.3f}")
print(f"    Dense-only (α=1.0)          : {mean_dense:.3f}")
print(f"    Improvement                 : {improvement:+.1f}%")
print()
print("  GENERATION QUALITY (RAGAS)")
print(f"    Faithfulness                : {mean_faithfulness:.3f}")
print(f"    Answer Relevancy            : {mean_answer_relevancy:.3f}")
print()
print("  DATASET")
print(f"    Total eval queries          : {len(EVAL_DATASET)}")
print(f"    Topic queries               : {len(topic_results)}")
print(f"    Negative queries            : {len(negative_results)}")
print(f"    RAGAS samples               : {len(ragas_rows)}")
print("=" * 60)
print()
print("Resume line:")
print(
    f"  Built a Retrieval-Augmented Generation (RAG) system for UTP past year "
    f"exam papers with hybrid dense-sparse retrieval and cross-encoder reranking; "
    f"evaluated across {len(EVAL_DATASET)} manually-labelled queries achieving "
    f"Precision@5 of {mean_hybrid:.0%} (vs {mean_dense:.0%} dense-only baseline), "
    f"Faithfulness {mean_faithfulness:.2f}, Answer Relevancy {mean_answer_relevancy:.2f}."
)

In [ ]:
# ── Export results to JSON ────────────────────────────────────────────────────
export = {
    "summary": {
        "precision_at_5_hybrid":   mean_hybrid,
        "precision_at_5_dense":    mean_dense,
        "improvement_pct":         improvement,
        "faithfulness":            mean_faithfulness,
        "answer_relevancy":        mean_answer_relevancy,
        "n_eval_queries":          len(EVAL_DATASET),
        "n_ragas_samples":         len(ragas_rows),
    },
    "per_query_precision": results,
    "per_query_ragas":     ragas_df[["user_input", "faithfulness", "answer_relevancy"]].to_dict(orient="records"),
}

Path("data").mkdir(exist_ok=True)
with open("data/evaluation_results.json", "w") as f:
    json.dump(export, f, indent=2)

print("✅ Results saved → data/evaluation_results.json")